# NLLB-200 Fine-Tuning: English → Twi Translation

Fine-tunes `facebook/nllb-200-distilled-600M` on pre-cleaned, pre-split data.

**Model:** NLLB-200 (600M params, distilled)
**Platform:** Kaggle P100 (16 GB VRAM)
**Direction:** `eng_Latn` → `twi_Latn`

> Data cleaning and train/val/test splitting were done in `01_data_cleaning.ipynb`.
> This notebook loads the ready-to-use CSVs directly — no re-splitting needed.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets sacrebleu sentencepiece \
             accelerate evaluate

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — English → Twi Direction
# ═══════════════════════════════════════════════════════════════

MODEL_ID  = "facebook/nllb-200-distilled-600M"
SRC_LANG  = "eng_Latn"   # English (source)
TGT_LANG  = "twi_Latn"   # Twi (target)
MAX_LEN   = 256

# Training
NUM_EPOCHS       = 10
BATCH_SIZE       = 8
GRAD_ACCUM       = 4      # Effective batch = 32
LEARNING_RATE    = 3e-5
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01
LABEL_SMOOTHING  = 0.1

# Paths — pre-cleaned CSVs from 01_data_cleaning.ipynb
DATA_DIR        = "/kaggle/input/twi-en-cleaned-data"   # update if different
TRAIN_CSV       = f"{DATA_DIR}/train_en_to_twi.csv"
VAL_CSV         = f"{DATA_DIR}/val_en_to_twi.csv"
TEST_CSV        = f"{DATA_DIR}/test_en_to_twi.csv"

OUTPUT_DIR      = "./en-twi-nllb-v2"
HF_REPO         = "ninte/en-twi-nllb-v2"  # ← your HF username

print("Configuration loaded.")
print(f"  Model     : {MODEL_ID}")
print(f"  Direction : {SRC_LANG} → {TGT_LANG}")
print(f"  Eff. batch: {BATCH_SIZE * GRAD_ACCUM}")

## 3. Load Pre-Cleaned Data

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Sanity check columns
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert 'twi' in df.columns and 'english' in df.columns, \
        f"{name}: expected 'twi' and 'english' columns, got {list(df.columns)}"
    assert df[['twi','english']].isna().sum().sum() == 0, \
        f"{name}: unexpected nulls found"

print(f"Train : {len(train_df):,} pairs")
print(f"Val   : {len(val_df):,} pairs")
print(f"Test  : {len(test_df):,} pairs")
print(f"Total : {len(train_df)+len(val_df)+len(test_df):,} pairs")
print("\nSample:")
train_df.sample(3, random_state=42)

## 4. Load NLLB Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"Device      : {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
if torch.cuda.is_available():
    print(f"VRAM used   : {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 5. Tokenize Datasets (English → Twi)

Source is English, target is Twi — the opposite of notebook 02.

In [ ]:
from datasets import Dataset

def tokenize_batch(batch):
    """Tokenize for NLLB English → Twi direction."""
    tokenizer.src_lang = SRC_LANG  # eng_Latn

    # Source: English
    model_inputs = tokenizer(
        batch["english"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    # Target: Twi
    labels = tokenizer(
        text_target=batch["twi"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['twi','english']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['twi','english']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[['twi','english']].reset_index(drop=True))

print("Tokenizing (English → Twi)...")
train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=256,
                          remove_columns=['twi','english'])
val_tok   = val_ds.map(tokenize_batch,   batched=True, batch_size=256,
                          remove_columns=['twi','english'])
test_tok  = test_ds.map(tokenize_batch,  batched=True, batch_size=256,
                          remove_columns=['twi','english'])

print("Tokenization complete.")
print(f"Sample keys: {list(train_tok[0].keys())}")

## 6. Define Evaluation Metrics

In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric  = evaluate.load("ter")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.argmax(preds, axis=-1) if preds.ndim == 3 else preds
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = bleu_metric.compute(predictions=decoded_preds,
                                references=[[r] for r in decoded_labels])
    chrf = chrf_metric.compute(predictions=decoded_preds,
                                references=decoded_labels)
    ter  = ter_metric.compute(predictions=decoded_preds,
                               references=[[r] for r in decoded_labels])
    return {
        "bleu": round(bleu["score"], 2),
        "chrf": round(chrf["score"], 2),
        "ter":  round(ter["score"],  2),
    }

print("Metrics ready: BLEU, chrF, TER")

## 7. Training

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import os

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    label_smoothing_factor=LABEL_SMOOTHING,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Trainer ready  — {len(train_df):,} training pairs")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ── Checkpoint resume ──────────────────────────────────────────
checkpoints = []
if os.path.exists(OUTPUT_DIR):
    checkpoints = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("checkpoint-")]

if checkpoints:
    latest      = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from = os.path.join(OUTPUT_DIR, latest)
    print(f"Resuming from: {resume_from}")
else:
    resume_from = None
    print("Starting fresh training run")

trainer.train(resume_from_checkpoint=resume_from)

## 8. Full Evaluation on Test Set

In [ ]:
import sacrebleu as sb

def full_evaluate_en_twi(model, tokenizer, df, label=''):
    """Evaluate English → Twi. Source=english col, ref=twi col."""
    model.eval()
    tgt_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)  # twi_Latn
    preds, refs  = [], []

    for _, row in df.iterrows():
        tokenizer.src_lang = SRC_LANG  # eng_Latn
        inputs = tokenizer(row['english'], return_tensors="pt",
                           truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_token_id,
                num_beams=4, max_length=MAX_LEN,
                no_repeat_ngram_size=3, length_penalty=1.0, early_stopping=True,
            )
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row['twi'])  # reference is Twi

    bleu = sb.corpus_bleu(preds, [refs])
    chrf = sb.corpus_chrf(preds, [refs])
    ter  = sb.corpus_ter(preds, [refs])

    print(f"\n{'='*55}")
    print(f"  {label} ({len(df):,} pairs)")
    print(f"{'='*55}")
    print(f"  BLEU   : {bleu.score:.2f}")
    print(f"  chrF   : {chrf.score:.2f}")
    print(f"  TER    : {ter.score:.2f}  (lower is better)")
    print(f"  BLEU 1g: {bleu.precisions[0]:.2f}")
    print(f"  BLEU 4g: {bleu.precisions[3]:.2f}")
    print(f"  BP     : {bleu.bp:.4f}")
    print(f"{'='*55}\n")
    return {"bleu": bleu.score, "chrf": chrf.score, "ter": ter.score,
            "preds": preds, "refs": refs}

results = full_evaluate_en_twi(model, tokenizer, test_df,
                                label="English → Twi Test Set")

# Baseline comparison
print("  v1 MarianMT baseline:")
print("  En→Twi  BLEU  6.32 | chrF 28.74 | TER 91.55")
delta_bleu = results['bleu'] - 6.32
delta_chrf = results['chrf'] - 28.74
print(f"  Improvement: BLEU {delta_bleu:+.2f} | chrF {delta_chrf:+.2f}")

## 9. Translation Examples

In [ ]:
def translate(text):
    tokenizer.src_lang = SRC_LANG
    tgt_id  = tokenizer.convert_tokens_to_ids(TGT_LANG)
    inputs  = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    inputs  = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=tgt_id,
                              num_beams=4, max_length=MAX_LEN,
                              no_repeat_ngram_size=3, length_penalty=1.0,
                              early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print("=" * 70)
print("TRANSLATION EXAMPLES — English → Twi")
print("=" * 70)
for _, row in test_df.sample(5, random_state=1).iterrows():
    print(f"\nEnglish: {row['english']}")
    print(f"Pred   : {translate(row['english'])}")
    print(f"Ref    : {row['twi']}")

## 10. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model.config.update({
    "src_lang": SRC_LANG,
    "tgt_lang": TGT_LANG,
    "language": ["en", "tw"],
    "tags": ["translation", "twi", "akan", "ghana", "low-resource", "nllb", "en-tw"],
    "license": "cc-by-nc-4.0",
})

print(f"Pushing to {HF_REPO}...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"\nPublished: https://huggingface.co/{HF_REPO}")

## 11. Training Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING SUMMARY — English → Twi")
print("="*70)
print(f"  Base model     : {MODEL_ID}")
print(f"  Direction      : {SRC_LANG} → {TGT_LANG}")
print(f"  Train pairs    : {len(train_df):,}")
print(f"  Val pairs      : {len(val_df):,}")
print(f"  Test pairs     : {len(test_df):,}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Max seq length : {MAX_LEN}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Best metric    : chrF (primary for Twi)")
print(f"\n  Test BLEU      : {results['bleu']:.2f}  (v1 baseline:  6.32)")
print(f"  Test chrF      : {results['chrf']:.2f}  (v1 baseline: 28.74)")
print(f"  Test TER       : {results['ter']:.2f}  (v1 baseline: 91.55)")
print(f"\n  Published      : https://huggingface.co/{HF_REPO}")
print("="*70)